# Introduction to Trace DG

## Prerequisites

### What is Trace DG?

Trace DG methods are used to simulate partial differential equations (PDEs) on surfaces.
In BoSSS, this surface is given by the Level-Set.
The core idea behind trace methods is to reformulate the surface PDE as a problem defined on a surrounding or nearby volume, 
which is then solved using operation already available in the extended discontinuous Galerkin (XDG) method.
While the property-of-interest, e.g., some concentration of a surface-active substance (aka. surfactant)
is defined in the entire part of the mesh which is cut by the interface,
only the trace of the solution on the surface has a physical interpretation.

Further reading:
- Utz, T. (2018). Level set methods for high-order unfitted discontinuous Galerkin schemes. 
  PhD thesis, TU Darmstadt
- Fischer, T. (2023). Numerical Investigation of partial differential equations on embedded manifolds employing Trace-methods.
  Master’s thesis, TU Darmstadt
- Larson, Burman, Hansbo, and Massing. A cut discontinous Galerkin method for the Laplace-Beltrami operator.
  In: Journal of Numerical Analysis 37.1 (2017), pp. 138–169
- Ulfsby, Massing and Sticko. Stabilized cut discontinous Galerkin methods for advection-reaction problems on surfaces.
  In: Comput. Methods Appl. Mech. Engrg. 413 (2023)

### Loading the BoSSS-library

Note: 
1. This tutorial can be found in the source code repository as as `TraceDG.ipynb`. 
   One can directly load this into Jupyter to interactively work with the following code examples.
2. **In the following line, the reference to `BoSSSpad.dll` is required**. 
   You must either set `#r "BoSSSpad.dll"` to something which is appropirate for your computer
   (e.g. `C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll` if you installed the binary distribution),
   or, if you are working with the source code, you must compile `BoSSSpad` and put it side-by-side to this worksheet file
   (from the original location in the repository, you can use the scripts `getbossspad.sh`, resp. `getbossspad.bat`).




In [1]:
#r "C:\Users\flori\Documents\BoSSS-premaster\public\src\L4-application\BoSSSpad\bin\Debug\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Defining a Trace DG field

In order to define/instantiate a `TraceDG` field, the following objects are required:
1. A background mesh, on which the trace-field will be embedded
2. A level-set-field which defines the interface.
   Furthermore, the so-called level-set tracker is required, which performs some necessary "bookkeeping" on the cut-cells.
3. A polynomial basis for the trace DG field (class `TraceDGBasis`)
4. Finally, the `TraceDG` field can be instantiated 



For sake of simplicity, a we use a 2D Cartesian background mesh (in BoSSS, also called a grid):

In [2]:
var grd = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1, 1, 50), GenericBlas.Linspace(-1, 1, 50));

The manifold/interface on which the trace should be defined will just be a circle with radius 0.7.
This can be described as the zero-iso-surface of the form $\varphi := 0.7^2 - x^2 - y^2$.
Such an interface can be described **exactly**, i.e., up to round-of-errors, by a DG-Level-Set of polynomial degree 2.

In [4]:
LevelSet phi = new LevelSet(new Basis(grd, 2), "Phi"); // creates a level-set of DG polynomial degree 2
phi.ProjectField((x,y) => 0.7*0.7 - x*x - y*y); // set this to the respective field

In [ ]:
var lsTrk = new LevelSetTracker(phi); // bookkeping of cut-cells, etc.